# 01 · Policy ingestion and the Q&A dataset
**Travel Shield Policy Explainer - capstone.** This notebook turns the 47-page policy PDF into searchable chunks and explores the trilingual input/output dataset built from it.

Steps: PDF → clean pages → section-aware chunks (page + heading) → dataset EDA.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from src import config
pd.set_option("display.max_colwidth", 120)

## 1. PDF → pages → section-aware chunks
Footers and page headers are removed; each chunk keeps the heading it falls under so answers can cite *[Section 14, p. 27]*.

In [ ]:
from src.ingest import extract_pages, build_chunks, save_chunks
pages = extract_pages()
chunks = build_chunks(pages)
save_chunks(chunks)
print(len(pages), "pages ->", len(chunks), "chunks")
pd.DataFrame(chunks)[["chunk_id", "page", "section"]].sample(8, random_state=1)

In [ ]:
ch = pd.DataFrame(chunks)
ch["chars"] = ch.text.str.len()
print(ch.chars.describe().round(0))
ch.section.str.replace(r" - .*", "", regex=True).value_counts().head(12)

## 2. The input/output dataset
Built by `scripts/build_dataset.py` from hand-written items in `data/qa/seed_part*.py`. Every item cites its section and page, and has the same question **and** answer in English, Tagalog and Taglish.

In [ ]:
qa = pd.read_csv(config.QA_FILE)
clf = pd.read_csv(config.CLASSIFIER_FILE)
print(f"{qa.item_id.nunique()} items, {len(qa)} Q&A pairs, {len(clf)} classifier rows")
qa.pivot_table(index="topic", columns="language", values="qa_id", aggfunc="count").assign(total=lambda d: d.sum(axis=1))

In [ ]:
qa[qa.item_id == "BAG-04"][["language", "input", "output"]]

In [ ]:
qa["q_words"] = qa.input.str.split().str.len()
qa["a_words"] = qa.output.str.split().str.len()
qa.groupby("language")[["q_words", "a_words"]].mean().round(1)

## 3. Split
Grouped by item (≈20% of each topic's items in test). No translation of a test question appears in training, so the evaluation measures generalisation to *new* questions.

In [ ]:
qa.groupby(["split", "language"]).size().unstack()

## 4. Chat-format JSONL
`train.jsonl` / `test.jsonl` hold system/user/assistant records, usable as few-shot examples or for supervised fine-tuning.

In [ ]:
print(open(config.DATA_QA / "train.jsonl", encoding="utf-8").readline()[:600])